# MOD10 · Class 3 — Classification with KNN


## Step 0 – Execute code



---

importing the database

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://raw.githubusercontent.com/Adrien-Rouillon/Machine-learning---SpotifyDB/refs/heads/main/spotify_artist_streaming_2020_2025.csv"
df = pd.read_csv(url)


The function below generates errors into the DB thanks to an AI algorithm.

In [11]:
def add_noise(df, seed=42):  # return a messy copy of a clean DataFrame
    rng = np.random.default_rng(seed)  # fixed seed -> same mess every run
    out = df.copy(deep=True)  # work on a deep copy so df is never modified
    num_cols = out.select_dtypes("number").columns.tolist()  # numeric columns available to corrupt
    if len(num_cols) < 2:  # need at least two numeric columns
        raise ValueError("df needs at least 2 numeric columns")  # fail loudly instead of guessing
    miss_cols = num_cols[:2]  # first two numeric columns get missing values
    out_col = num_cols[0]  # first numeric column gets outliers
    text_col = num_cols[2] if len(num_cols) > 2 else num_cols[1]  # third (or second) becomes text
    n = len(out)  # number of original rows

    # --- extreme outliers ---
    out_idx = rng.choice(out.index, size=3, replace=False)  # pick 3 rows to corrupt
    col = out[out_col].astype(float)  # float copy so large values fit
    col.loc[out_idx] = col.mean() + 20 * col.std() * rng.choice([-1, 1], size=3)  # push Â±20 std from the mean
    out[out_col] = col  # write outliers back

    # --- ~5% missing values in two numeric columns ---
    for c in miss_cols:  # handle each target column
        nan_idx = rng.choice(out.index, size=max(1, round(0.05 * n)), replace=False)  # 5% random rows
        out[c] = out[c].astype(float)  # ints can't hold NaN, so cast to float
        out.loc[nan_idx, c] = np.nan  # blank them out

    # --- inconsistent categorical column ---
    base = rng.choice(["Red", "Green", "Blue"], size=n)  # clean underlying categories
    variants = [str.lower, str.upper, str.title, lambda s: f" {s}", lambda s: f"{s}  "]  # case/spacing styles
    out["color"] = [variants[rng.integers(len(variants))](v) for v in base]  # apply a random style per row

    # --- numeric column stored as text with a stray 'unknown' ---
    out[text_col] = out[text_col].map(lambda v: "" if pd.isna(v) else str(v)).astype(object)  # numbers -> strings
    out.loc[rng.choice(out.index), text_col] = "unknown"  # one non-numeric token

    # --- 5 exact duplicate rows ---
    dup_rows = out.loc[rng.choice(out.index, size=5, replace=False)]  # copy 5 existing (already messy) rows
    out = pd.concat([out, dup_rows], ignore_index=True)  # append them -> exact duplicates
    out = out.sample(frac=1, random_state=seed).reset_index(drop=True)  # shuffle so dupes aren't at the end

    return out  # the messy copy

df = add_noise(df)

Now that we've added noise to the DB, we need to clean it.

First, we remove duplicate rows and where popularity is missing.

Then, we look at the duration of each music. We keep it if the value is between 30k (30 sec) and 900k (15 min), or if the value is missing. We drop the rest.

The reason we kept the missing data is to fill it with the median of the other value.

Finally, we convert danceability to a numeric type, and any value that can’t be parsed becomes NaN. We can thus drop every row without a number easily.

In [12]:

print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()


df = df.dropna(subset=["popularity"])

df = df[df["duration_ms"].isna() | df["duration_ms"].between(30_000, 900_000)]

df["duration_ms"] = df["duration_ms"].fillna(df["duration_ms"].median())

df["danceability"] = pd.to_numeric(df["danceability"], errors="coerce")
df = df.dropna(subset=["danceability"])

print(df.shape)
print("popularity max:", df["popularity"].max())


Duplicates: 5
(47497, 34)
popularity max: 92.0


We then drop all unnecessary columns, and transform the rest into numeric values thanks to get_dummies() and map(), so the data can be scaled.

In [13]:
to_drop = [
    "track_name", "album_name", "artist_name", "release_date",              #text with unique value
    "popularity_category",                                                  #Leak : popularity drift
    "duration_minutes", "key_name", "mode_name", "is_explicit_bool",
    "loudness_category", "release_quarter", "log_stream_count",             #doubles
    "index", "track_id","color"
]

df = df.drop(columns=[c for c in to_drop if c in df.columns])

bool_map = {True:1,False:0,"true":1,"false":0,"True":1,"False":0}
df['explicit'] = df['explicit'].map(bool_map)
df['is_weekend_release'] = df['is_weekend_release'].map(bool_map)

for c in ["genre", "country", "release_day_of_week", "label"]:
    df[c] = df[c].map(str.lower)

top_labels = set(df["label"].value_counts().nlargest(10).index)
df["label"] = df["label"].map(lambda x: x if x in top_labels else "other")

df["key"] = df["key"].map(lambda x: f"key_{x}")

df = pd.get_dummies(df, columns=["key", "genre", "country", "release_day_of_week", "label"],
                    drop_first=True, dtype=int)

## Step 1 – Creation of categories



---

Before we do anything, we must first define our goal, and create categories so that the machine can classify the data.
We decided to use popularity as our target column.

But why? Our goal is to determine if a music is a hit or not. And the popularity is the closest data we can link or define to a hit: if the popularity is over a threshold, then we can conclude that it's a hit. Howerver if it's below said threshold, then we can conclude that it isn't.

For this project, we decide arbitrarly that the threshold is at the top 10% in popularity. We can use quantile to search the percent we want.


In [14]:
print(df["popularity"].quantile([0.5, 0.75, 0.9, 0.95]))

0.50    26.0
0.75    39.0
0.90    50.0
0.95    58.0
Name: popularity, dtype: float64


We choose 50 as the threshold.

Then we can split all the data :

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

THRESHOLD = 50

y = (df["popularity"] >= THRESHOLD).astype(int) # popularity >= threshold : 0 = unpopular, 1 = popular
X = df.drop(columns=["popularity"])

## Step 2 – Features and label



---

In this part, we have to see if our classes are equal.

In [16]:
print(y.value_counts())
print(y.value_counts(normalize=True).round(3))

popularity
0    42354
1     5143
Name: count, dtype: int64
popularity
0    0.892
1    0.108
Name: proportion, dtype: float64


As we can see, our values are imbalanced. This is due to the definition of our hit. We have 10% of hit and 90% of non hit. The consequence is that accuracy alone will be misleading: a model that always predicts "not a hit" would already reach 90% accuracy. We will therefore look at the confusion matrix and especially at how many real hits the model finds.

## Step 3 – Split and trainning



---

We then split the data into a training set (80%) and a test set (20%).
Because hits are rare, we need both sets to keep the same proportion of hits.

We then scale the features since KNN relies on distances and a feature with large values would otherwise dominate the others.

The scaler is fitted on the training set only, and the test set is transformed with the training mean and standard deviation. This prevents data leakage, since no information from the test set influences the training.

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print("Part of hits train/test :", y_train.mean().round(3), y_test.mean().round(3))

(37997, 89) (9500, 89)
Part of hits train/test : 0.108 0.108


In [18]:
scaler = StandardScaler().fit(X_train)       # fit on TRAIN only
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

## Step 4 – Training KNN



---

Now we can test with the 10 first values of the trainning. As you can see, all the values are not a hit (we have 10% chance to have a hit so it's logical)

In [20]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=11)
model.fit(X_train_s, y_train)

preds = model.predict(X_test_s)
preds[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

We will now try to find the number of neighbor the most adapted for our goal.

In [21]:
from sklearn.metrics import accuracy_score, recall_score
for k in [1, 3, 5, 11, 21, 51, 101, 195]:
    p = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train).predict(X_test_s)
    print(k, "acc", round(accuracy_score(y_test, p), 4), "recall_hit", round(recall_score(y_test, p), 3))

1 acc 0.8316 recall_hit 0.168
3 acc 0.8796 recall_hit 0.085
5 acc 0.8927 recall_hit 0.065
11 acc 0.8956 recall_hit 0.04
21 acc 0.8954 recall_hit 0.034
51 acc 0.8947 recall_hit 0.028
101 acc 0.8943 recall_hit 0.024
195 acc 0.894 recall_hit 0.021


As you can see in this result, the more there are k, the more there is a high accuracy. But this evolution stops at 11 and then decreases. Furthermore recall_hit decreases aswell, so 11 is the best choice.

However, recall for hits is very low: at k = 11, the model detects only 4% of the real hits and misses the other 96%.

## Step 5 – Evaluate



---

In [22]:
from sklearn.metrics import accuracy_score, confusion_matrix

print("accuracy", accuracy_score(y_test,preds))

print(confusion_matrix(y_test,preds))

accuracy 0.895578947368421
[[8467    4]
 [ 988   41]]


This is our accuracy and our confusion matrix. As we can see, the accuracy is gret, but there is more to it. The confusion matrix made 4 errors predicting hits, and 988 errors trying to predict non hits.

## Step 6 – Interpretation



---

In this configuration, accuracy is really tricky because even if it's high, this is due to the majority of non hits and the rarity of hits (A system that always says non hit will be most optimal than a system that try a little bit more to find a hit). The ecart between a system that always choose non hit and our system is 0,8956 - 0,8917 = 0.04. As we can see in recall_hits datas, the IA find only 4% of the hits and that is really bad because it is one of the most representative data for the efficacity of this system (recall_hit are more usefull than accurracy in this contexte). If we want to optimise it, we should make a ponderation of the neighbor due to their distances, take less columns because there is to much data, ...

## Step 7 – Model fit



---

KNN does not seem well suited to this project: it finds only 4 % of the hits, because the hit class is rare and the distances are diluted over 89 columns. A tree-based model (decision tree / random forest) could handle the imbalance better (`class_weight="balanced"`) and would show which variable is the most linked to popularity.